# 교차 엔트로피 실습 (25분)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/slides/v2/ml1/week02/cross_entropy/lab/cross_entropy.ipynb)

정답에 건 확률 $q$ 에 대해 벌점 $-\log q$. 슬라이드의 칩 비유를 코드와 진짜 데이터로 확인한다.

| 파트 | 내용 | 시간 |
|---|---|---|
| 1 | 칩 비유를 코드로: 벌점 손계산 → scikit-learn `log_loss` 와 맞추기 | 7분 |
| 2 | 유방암 진단 데이터: 로지스틱회귀를 교차 엔트로피 vs MSE 로 학습 | 10분 |
| 3 | 셰익스피어: 다음 글자에 칩 나눠 걸기 (고르게 vs 세어서) | 8분 |

`None` 으로 비워 둔 `TODO` 줄을 채우고 셀을 차례로 실행하세요.

In [ ]:
import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
_have = {f.name for f in fm.fontManager.ttflist}
_font = next((f for f in ["Noto Sans CJK KR", "NanumGothic", "AppleGothic", "Malgun Gothic"] if f in _have), None)
if _font is None:   # Colab 등 한글 글꼴이 없으면 나눔고딕을 받아 등록
    try:
        _p = "/tmp/NanumGothic.ttf"
        if not os.path.exists(_p):
            urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf", _p)
        fm.fontManager.addfont(_p); _font = fm.FontProperties(fname=_p).get_name()
    except Exception as e:
        print("한글 글꼴 설치 실패 (그래프 글자만 깨지고 실습은 가능):", e)
if _font:
    plt.rcParams["font.family"] = _font
plt.rcParams["axes.unicode_minus"] = False
from sklearn.metrics import log_loss

## 파트 1. 칩 비유를 코드로

정답 쪽에 남은 칩 비율이 $q$ 일 때 벌점(비트) $= -\log_2 q$. 슬라이드의 값 (90개 0.15 · 50개 1 · 1개 6.64) 이 나오는지 확인한다.

In [ ]:
def penalty_bits(q):
    """정답에 건 비율 q (0~1) → 벌점(비트)"""
    return None             # TODO: -log₂ q

for left in [90, 50, 25, 1]:
    print(f"정답 쪽 칩 {left:3d}개 → 벌점 {penalty_bits(left / 100):.2f} 비트")
with np.errstate(divide="ignore"):
    print("정답 쪽 칩   0개 → 벌점", penalty_bits(0.0), "(파산)")

실제 학습 코드는 자연로그 $\ln$ 을 쓴다. 이진 분류의 평균 교차 엔트로피:
$$L = -\frac{1}{m}\sum_{i=1}^{m} \big[y_i \ln \hat y_i + (1-y_i)\ln(1-\hat y_i)\big]$$

In [ ]:
def binary_ce(y, p):
    """y: 정답 0/1 배열, p: 1일 확률 배열 → 평균 교차 엔트로피 (ln)"""
    return None   # TODO: 위 식

y = np.array([1, 0, 1, 1, 0])
p = np.array([0.9, 0.1, 0.5, 0.01, 0.9])
print("직접 계산:", round(binary_ce(y, p), 4))
print("sklearn  :", round(log_loss(y, p), 4))
print("각 샘플 벌점:", np.round(-np.log(np.where(y == 1, p, 1 - p)), 2))

**관찰**: 다섯 개 중 넷째(정답 1에 0.01)와 다섯째(정답 0인데 1에 0.9)가 평균 벌점의 대부분을 차지한다. 확신하고 틀린 샘플이 손실을 끌고 간다.

## 파트 2. 유방암 진단: 교차 엔트로피 vs MSE

같은 로지스틱회귀 $\hat y = \sigma(Xw + b)$ 를 두 가지 손실로 학습한다. 슬라이드의 기울기:
- 교차 엔트로피: $\dfrac{\partial L}{\partial z} = \hat y - y$
- MSE $(\hat y - y)^2$: $\dfrac{\partial L}{\partial z} = 2(\hat y - y)\,\hat y(1-\hat y)$

**확신하고 틀린 상태**에서 출발시켜 (좋은 방향의 반대로 초기화) 무엇이 달라지는지 본다.

In [ ]:
from sklearn.datasets import load_breast_cancer

d = load_breast_cancer()
X = (d.data - d.data.mean(0)) / d.data.std(0)            # 특징 30개 표준화
y = d.target.astype(float)                               # 1 = 양성(benign), 0 = 악성(malignant)
m, n = X.shape
print(f"{m}명, 특징 {n}개, 양성 비율 {y.mean():.0%}")

sigmoid = lambda z: 1 / (1 + np.exp(-z))

def grad(w, b, loss):
    p = sigmoid(X @ w + b)
    if loss == "ce":
        dz = None               # TODO: 교차 엔트로피의 ∂L/∂z
    else:
        dz = None               # TODO: MSE 의 ∂L/∂z
    return X.T @ dz / m, dz.mean()

def train(loss, w0, steps=300, lr=0.5):
    w, b = w0.copy(), 0.0
    ces, accs = [], []
    for _ in range(steps):
        p = np.clip(sigmoid(X @ w + b), 1e-12, 1 - 1e-12)
        ces.append(binary_ce(y, p)); accs.append(np.mean((p > 0.5) == y))
        gw, gb = grad(w, b, loss)
        w, b = w - lr * gw, b - lr * gb
    return ces, accs

good = X.T @ (y - y.mean()) / m                           # 대략 옳은 방향
w_wrong = -3 * good / np.linalg.norm(good)                # 그 반대로, 크게: 확신하고 틀린 출발점
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for loss, c in [("ce", "C0"), ("mse", "C1")]:
    ces, accs = train(loss, w_wrong)
    axs[0].plot(ces, c, label=loss); axs[1].plot(accs, c, label=loss)
    print(f"{loss:3s}: 처음 정확도 {accs[0]:.0%} → 20걸음 {accs[20]:.0%} → 300걸음 {accs[-1]:.0%}")
axs[0].set_title("교차 엔트로피 (평가용)"); axs[0].set_yscale("log"); axs[1].set_title("정확도")
for a in axs: a.set_xlabel("걸음"); a.legend()
plt.tight_layout(); plt.show()

**해 보기**: `w_wrong` 대신 `np.zeros(n)` (반반에서 출발) 으로 바꿔 보자. 이때도 차이가 클까? 슬라이드의 기울기 그림에서 $\hat y = 0.5$ 근처의 두 곡선을 비교해 보자.

## 파트 3. 셰익스피어: 다음 글자에 칩 나눠 걸기

언어 모델은 다음 글자(단어)에 칩을 나눠 걸고, 실제 다음 글자에 건 비율로 벌점을 받는다. 처음 실행하면 글 파일(약 1MB)을 내려받는다.
- 모델 1: 모든 글자에 **고르게** 건다
- 모델 2: 앞 글자를 보고, 학습 글에서 **세어 본** 비율대로 건다 (bigram)

In [ ]:
import urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/shakespeare.txt"):
    urllib.request.urlretrieve(url, "data/shakespeare.txt")
text = open("data/shakespeare.txt").read()
chars = sorted(set(text)); V = len(chars); idx = {c: i for i, c in enumerate(chars)}
cut = int(len(text) * 0.9)
train_txt, test_txt = text[:cut], text[cut:]
print(f"글자 {len(text):,}개, 종류 {V}개")
print(test_txt[:200])

In [ ]:
# 모델 1: 고르게 → 어떤 글자든 1/V
uniform_bits = penalty_bits(1 / V)
print(f"고르게 걸기: 글자당 벌점 {uniform_bits:.2f} 비트")

# 모델 2: 앞 글자 a 다음에 b 가 나온 횟수 (+1 스무딩) 로 비율을 정한다
counts = np.ones((V, V))
for a, b in zip(train_txt, train_txt[1:]):
    counts[idx[a], idx[b]] += 1
q = counts / counts.sum(1, keepdims=True)                 # q[a, b] = a 다음에 b 에 건 비율

pairs = np.array([(idx[a], idx[b]) for a, b in zip(test_txt, test_txt[1:])])
bits = None   # TODO: 실제 다음 글자에 건 비율의 벌점들
print(f"세어서 걸기: 글자당 평균 벌점 {bits.mean():.2f} 비트")

In [ ]:
# 앞 글자가 'q' 일 때 어디에 걸었나? (영어에서 q 다음은 거의 u)
top = np.argsort(q[idx['q']])[::-1][:3]
for t in top:
    print(repr(chars[t]), f"{q[idx['q'], t]:.0%}")

**확인 질문**
1. 파트 1 에서 정답에 0 을 건 샘플이 하나라도 있으면 평균 교차 엔트로피는? `binary_ce` 에 넣어 보자 (그래서 실전 코드는 `clip` 을 한다).
2. 파트 2 에서 MSE 가 확신하고 틀린 출발점에서 느린 이유를 슬라이드의 기울기 식으로 설명해 보자.
3. 파트 3 의 고르게 걸기 벌점이 왜 $\log_2 V$ 인가?
4. 파트 3 에서 앞 글자 두 개를 보고 세면(trigram) 벌점은 더 줄어들까? 학습 글에 한 번도 안 나온 조합은 어떻게 될까?

데이터 출처: Wolberg et al., Breast Cancer Wisconsin (Diagnostic), UCI (scikit-learn) · Tiny Shakespeare, Karpathy char-rnn.